<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Network Configuration and Access Policy

Network configuration and access policy determine which Duckiedrone connections are permitted. Treat applicable access controls, credentials, and device identities as safety boundaries rather than obstacles to bypass.

## Automatic address configuration and private addresses

DHCP can provide a device with an IP address, subnet prefix, gateway, and DNS servers automatically. The address is normally leased for a period of time, so a Duckiedrone's numeric IP can change when it reconnects or renews its lease. A DHCP server can reserve an address for a device, usually using its MAC address or a separate client identifier. A reservation is configured on the network, not by guessing an unused address on the Duckiedrone.

A route can include a protocol tag that suggests how it was installed, but it does not prove the full network configuration. For example, `proto dhcp` can indicate a route that the network manager installed from DHCP information. In this route, `via` identifies the default gateway and `src` gives the preferred source address for matching destinations:

```shell
default via 192.168.1.1 dev wlan0 proto dhcp src 192.168.1.201 metric 600
```

On many local networks, base stations and Duckiedrones use private IPv4 addresses: `10.0.0.0/8`, `172.16.0.0/12`, and `192.168.0.0/16`. These work inside the local network but are not routed directly across the public internet.

The private IPv4 address blocks and their full address ranges are listed in [Table 1](#table-1).

<table id="table-1" class="lx-table">
  <caption>Table 1: Private IPv4 address blocks.</caption>
  <thead>
    <tr>
      <th>Private IPv4 block</th>
      <th>Address range</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>10.0.0.0/8</code></td>
      <td><code>10.0.0.0</code> through <code>10.255.255.255</code></td>
    </tr>
    <tr>
      <td><code>172.16.0.0/12</code></td>
      <td><code>172.16.0.0</code> through <code>172.31.255.255</code></td>
    </tr>
    <tr>
      <td><code>192.168.0.0/16</code></td>
      <td><code>192.168.0.0</code> through <code>192.168.255.255</code></td>
    </tr>
  </tbody>
</table>

Two unrelated local networks can each contain a device at `192.168.1.201`. A private address therefore does not identify a Duckiedrone across the public internet, and the word *private* does not by itself mean that traffic is encrypted or trusted.

## Address translation and firewalls

A router joining a private network to the internet often performs Network Address Translation (NAT). For a connection started inside the local network, NAT records a temporary mapping so replies return to the correct private device. This allows many local devices to share a public IP. An unsolicited internet connection normally has no mapping and cannot reach a Duckiedrone without an explicitly configured rule.

An example NAT mapping for a Duckiedrone package request is shown in [Figure 1](#figure-1).

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    Duckiedrone request from 192.168.1.201
      |
      v
    router translates the private source address
      |
      v
    package server receives the request
      |
      v
    router maps the reply back
      |
      v
    Duckiedrone receives the response
  </pre>
  <figcaption>Figure 1: NAT mapping for an outbound package request.</figcaption>
</figure>

This outgoing connection does not make the Duckiedrone dashboard reachable from the public internet. NAT translates addresses; a firewall decides which traffic to allow or block. A router can perform both roles, but they are separate functions.

A __firewall__ applies traffic rules, often based on addresses, protocols, and ports. It can block ICMP echo replies used by `ping` or TCP port `22` used by SSH even when the hostname, IP, and route are correct. NAT and a firewall are different concepts even though home and shared-network routers often perform both roles. Keep protections in place and follow the documented access policy; do not change a rule merely to make a connection work.

Uncomplicated Firewall (UFW) is a tool for managing host firewall rules. Its `ufw status` command reports whether UFW is active and, when readable, lists configured rules. UFW status is firewall-policy evidence only: it does not show whether an application is listening or prove a remote connection can succeed. Firewall policy can matter when attaching a Duckiedrone to a Duckiematrix Entity because it can block communication with the Duckiematrix Engine; the [Duckietown Manual's guidance on attaching virtual robots](https://docs.duckietown.com/ente/duckietown-manual/50-duckiematrix/virtual-duckietown-robots/introduction-to-virtual-duckiebots.html?highlight=ufw#attaching-virtual-robots) gives that operational context.

For example, a documented UFW status snapshot can look like:

```shell
Status: active

To                         Action      From
--                         ------      ----
22/tcp                     ALLOW       192.168.0.0/24
```

This snapshot shows firewall-policy evidence: inbound TCP port `22` is allowed from the listed private subnet. It does not establish that SSH is listening, that the route works, or that a particular connection is authorized.

On systems that require administrator privileges to read this state, a normal-user attempt reports an error. As [Notebook 2's technical foreword](./2-linux-shell-and-navigation.ipynb#a-technical-foreword---do-not-use-sudo) explains, this LX does not ask you to use `sudo`. Therefore, do not attempt to run `ufw` for this LX.

## Work within explicit access policy

A network can deliberately isolate connected clients. For example, a base station and Duckiedrone on the same guest Wi-Fi network might both reach the internet while being unable to communicate with each other. Changing the Duckiedrone address would not remove that restriction.

A network used for the activity can require MAC registration, a particular wireless network or wired port, a network group, and firewall or multicast restrictions. Before connecting a Duckiedrone, confirm the following from the activity or network documentation:

- The network path, wireless network name, or wired location authorized for the activity.

- Whether the base station and Duckiedrone need registration or the same network group.

- Whether the planned connection method, such as local name resolution or SSH, is permitted.

- Relevant troubleshooting documentation and the information to record if a connection fails.

Use the smallest authorized diagnostic check: a known Duckiedrone hostname or address and one documented service port. Do not scan address ranges or arbitrary ports, change a device MAC identity, alter router or firewall settings, create an unauthorized hotspot or network bridge, or capture other people's traffic. A failed test is evidence to record and compare with the documented policy, not a reason to broaden the investigation.

### Try it

Inspect, but do not change, the current address and route configuration in an authorized shell:

```bash
ip -brief address
ip route
```

Record whether one assigned IPv4 address is private and whether the route table has a default gateway. Then identify one access-policy fact you must confirm before connecting to a physical Duckiedrone on that network.

<details>
<summary>Check your result</summary>

Private addresses commonly begin with `10.`, `172.16.` through `172.31.`, or `192.168.`, but an address alone does not establish authorization. The route table describes forwarding choices; it does not grant permission to access a device or change network configuration.

</details>

Keep passwords, tokens, private keys, and device-specific configuration out of notebooks, screenshots, chat messages, and issue reports. If SSH presents an unexpected host-key change or a browser presents a certificate warning, pause and verify the Duckiedrone's identity against a provisioning record or documented trusted fingerprint before accepting it. A virtual Duckiedrone is accessed locally from the base station, but traffic it sends still leaves through the base station and remains subject to policy. When recording an issue or using an available support channel, share only allowed network information, a known hostname, exact targeted command and output, and test time; omit credentials and sensitive configuration.

## Further reading

The IETF specifications for [DHCP](https://www.rfc-editor.org/rfc/rfc2131), [private IPv4 addresses](https://www.rfc-editor.org/rfc/rfc1918), and [traditional NAT](https://www.rfc-editor.org/rfc/rfc3022) provide protocol detail.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
